In [1]:
# load data
import pandas as pd

df = pd.read_csv('valorant-stats.csv')
df

,Name,Weapon Type,Price,Fire Rate,Wall Penetration,Magazine Capacity,HDMG_0,BDMG_0,LDMG_0,HDMG_1,BDMG_1,LDMG_1,HDMG_2,BDMG_2,LDMG_2
0,Classic,Sidearm,0,6.75,Low,12,78,26,22,78,26,22,66,22,18
1,Shorty,Sidearm,200,3.30,Low,2,36,12,10,24,8,6,9,3,2
2,Frenzy,Sidearm,400,10.00,Low,13,78,26,22,63,21,17,63,21,17
3,Ghost,Sidearm,500,6.75,Medium,15,105,33,26,88,25,21,88,25,21
4,Sheriff,Sidearm,800,4.00,High,6,160,55,47,160,55,47,145,50,43
5,Stinger,SMG,1000,18.00,Low,20,67,27,23,62,25,21,62,25,21
6,Spectre,SMG,1600,13.33,Medium,30,78,26,22,66,22,18,66,22,18
7,Bulldog,Rifle,2100,9.15,Medium,24,116,35,30,116,35,30,116,35,30
8,Guardian,Rifle,2500,6.50,Medium,12,195,65,49,195,65,49,195,65,49
9,Phantom,Rifle,2900,11.00,Medium,30,156,39,33,140,35,30,124,31,26


In [2]:
# EDA
print(f'Cek 5 data teraras \n {df.head()} \n')

print(f'Cek tipe data \n {df.dtypes} \n')

print(f'Cek missing values \n {df.isnull().sum()} \n')

print(f'Cek duplicate data \n {df.duplicated().sum()}')

Cek 5 data teraras 
       Name Weapon Type  Price  Fire Rate Wall Penetration  Magazine Capacity  \
0  Classic     Sidearm      0       6.75              Low                 12   
1   Shorty     Sidearm    200       3.30              Low                  2   
2   Frenzy     Sidearm    400      10.00              Low                 13   
3    Ghost     Sidearm    500       6.75           Medium                 15   
4  Sheriff     Sidearm    800       4.00             High                  6   

   HDMG_0  BDMG_0  LDMG_0  HDMG_1  BDMG_1  LDMG_1  HDMG_2  BDMG_2  LDMG_2  
0      78      26      22      78      26      22      66      22      18  
1      36      12      10      24       8       6       9       3       2  
2      78      26      22      63      21      17      63      21      17  
3     105      33      26      88      25      21      88      25      21  
4     160      55      47     160      55      47     145      50      43   

Cek tipe data 
 Name                    

In [6]:
numerical_columns = df.select_dtypes(include='number').columns

outlier_summary = []

for column in numerical_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]

    outlier_summary.append({
        'kolom': column,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'batas_bawah': lower_bound,
        'batas_atas': upper_bound,
        'jumlah_outlier': len(outliers),
        'persen_outlier': round(len(outliers) / len(df) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df

,kolom,Q1,Q3,IQR,batas_bawah,batas_atas,jumlah_outlier,persen_outlier
0,Price,800.0,2500.0,1700.0,-1750.00,5050.00,0,0.00
1,Fire Rate,3.5,10.0,6.5,-6.25,19.75,0,0.00
2,Magazine Capacity,6.0,25.0,19.0,-22.50,53.50,1,5.88
3,HDMG_0,72.0,156.0,84.0,-54.00,282.00,0,0.00
4,BDMG_0,26.0,39.0,13.0,6.50,58.50,3,17.65
5,LDMG_0,22.0,33.0,11.0,5.50,49.50,2,11.76
6,HDMG_1,63.0,156.0,93.0,-76.50,295.50,0,0.00
7,BDMG_1,22.0,39.0,17.0,-3.50,64.50,3,17.65
8,LDMG_1,18.0,33.0,15.0,-4.50,55.50,2,11.76
9,HDMG_2,63.0,145.0,82.0,-60.00,268.00,0,0.00


In [9]:
# handle outlier
def iqr_capping(df, columns):
    df = df.copy()

    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df[col] = df[col].clip(
            lower=lower,
            upper=upper
        )
    
    return df

In [10]:
# split data
x = df.drop(columns=['Price'])
y = df['Price']

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

In [12]:
# encoding
df = df.set_index('Name')
df['Wall Penetration'] = df['Wall Penetration'].map({'Low': 0, 'Medium': 1, 'High': 2})
df = pd.get_dummies(df, columns=['Weapon Type'], drop_first=True, dtype=int)

print(df.isna().sum().sum())   # harus 0
print(df.dtypes)               # pastikan nggak ada 'object'

0
Price                    int64
Fire Rate              float64
Wall Penetration         int64
Magazine Capacity        int64
HDMG_0                   int64
BDMG_0                   int64
LDMG_0                   int64
HDMG_1                   int64
BDMG_1                   int64
LDMG_1                   int64
HDMG_2                   int64
BDMG_2                   int64
LDMG_2                   int64
Weapon Type_Rifle        int64
Weapon Type_SMG          int64
Weapon Type_Shotgun      int64
Weapon Type_Sidearm      int64
Weapon Type_Sniper       int64
dtype: object
